# Projeto - API de Filmes com Flask

## Objetivo

Desenvolver uma API REST utilizando Flask para gerenciar um catálogo de filmes armazenado em um arquivo CSV.

O projeto deverá utilizar:

- Flask
- HTML (`render_template`)
- Arquivos CSV
- Métodos HTTP (GET, POST, PUT, PATCH e DELETE)

Arquivos fornecidos:

- `filmes_famosos.csv`
- `templates/home.html`

---

## 1. Carregamento dos dados

Ao iniciar a aplicação:

- Ler o arquivo `filmes_famosos.csv`
- Converter os registros para uma lista de dicionários
- Manter os dados em memória

---

## 2. Página inicial

Implementar a rota:

/

utilizando:

```python
render_template("home.html")
```

---

## 3. Rotas GET

### Listar todos os filmes

```text
GET /filmes
```

### Buscar filme por ID

```text
GET /filmes/<id>
```

### Buscar filme por nome

Exemplo:

```text
GET /filmes/busca?nome=interstellar
```

A busca deve ignorar diferenças entre maiúsculas e minúsculas.

### Listar filmes de um gênero

Exemplo:

```text
GET /filmes/genero/Drama
```

### Top 10 filmes

```text
GET /top10
```

Retornar os 10 filmes com maior nota IMDb.

### Estatísticas

```text
GET /estatisticas
```

Retornar um JSON contendo:

- quantidade total de filmes;
- média das notas IMDb;
- filme com maior nota;
- filme com menor nota;
- quantidade de filmes por gênero.

---

## 4. Cadastro de filmes

Implementar:

```text
POST /filmes
```

O novo filme deverá receber um ID automaticamente.

---

## 5. Atualização completa

Implementar:

```text
PUT /filmes/<id>
```

A atualização deve substituir todos os campos do filme.

---

## 6. Atualização parcial

Implementar:

```text
PATCH /filmes/<id>
```

A atualização deve modificar apenas os campos enviados na requisição.

---

## 7. Remoção

Implementar:

```text
DELETE /filmes/<id>
```

Remover o filme correspondente ao ID informado.

---

## 8. Persistência

Após qualquer operação de:

- POST
- PUT
- PATCH
- DELETE

o arquivo CSV deve ser atualizado para refletir as alterações realizadas.

# Gabarito

In [ ]:
from flask import Flask, jsonify, request, render_template
import pandas as pd

app = Flask(__name__)

In [ ]:
filmes = []

with open('../data/filmes_famosos.csv', encoding='utf-8') as arquivo:

    linhas = arquivo.readlines()

    for linha in linhas[1:]:  # pula cabeçalho

        dados = linha.strip().split(',')

        filme = {
            'id': int(dados[0]),
            'titulo': dados[1],
            'duracao': int(dados[2]),
            'genero': dados[3],
            'sinopse': dados[4],
            'nota_imdb': float(dados[5])
        }

        filmes.append(filme)

In [ ]:
def salvar_filmes():
    pd.DataFrame(filmes).to_csv(
        "filmes_famosos.csv",
        index=False
    )


def buscar_filme_por_id(id_filme):

    for filme in filmes:
        if filme["id"] == id_filme:
            return filme

    return None


def proximo_id():

    if not filmes:
        return 1

    return max(f["id"] for f in filmes) + 1

In [ ]:
@app.route('/')
def home():
    return render_template('home.html')

In [ ]:
@app.route('/filmes')
def listar_filmes():
    return jsonify(filmes)

In [ ]:
@app.route('/filmes/<int:id_filme>')
def buscar_por_id(id_filme):

    filme = buscar_filme_por_id(id_filme)

    if filme:
        return jsonify(filme)

    return jsonify({'erro':'Filme não encontrado'}), 404

In [ ]:
@app.route('/filmes/busca')
def buscar_por_nome():

    nome = request.args.get("nome")

    if not nome:
        return jsonify({"erro":"Informe o nome"}), 400

    resultado = []

    for filme in filmes:
        if nome.lower() in filme["titulo"].lower():
            resultado.append(filme)

    return jsonify(resultado)

In [ ]:
@app.route('/filmes/genero/<genero>')
def buscar_por_genero(genero):

    resultado = []

    for filme in filmes:
        if filme["genero"].lower() == genero.lower():
            resultado.append(filme)

    return jsonify(resultado)

In [ ]:
@app.route('/top10')
def top10():

    ordenado = sorted(
        filmes,
        key=lambda x: x["nota_imdb"],
        reverse=True
    )

    return jsonify(ordenado[:10])

In [ ]:
@app.route('/estatisticas')
def estatisticas():

    quantidade = len(filmes)

    nota_media = round(
        sum(f["nota_imdb"] for f in filmes) / quantidade,
        2
    )

    duracao_media = round(
        sum(f["duracao"] for f in filmes) / quantidade,
        1
    )

    melhor = max(
        filmes,
        key=lambda x: x["nota_imdb"]
    )

    return jsonify({
        "quantidade_filmes": quantidade,
        "nota_media": nota_media,
        "duracao_media": duracao_media,
        "melhor_filme": melhor["titulo"]
    })

In [ ]:
@app.route('/filmes', methods=['POST'])
def adicionar_filme():

    dados = request.get_json()

    novo_filme = {
        "id": proximo_id(),
        "titulo": dados["titulo"],
        "duracao": dados["duracao"],
        "genero": dados["genero"],
        "sinopse": dados["sinopse"],
        "nota_imdb": dados["nota_imdb"]
    }

    filmes.append(novo_filme)

    salvar_filmes()

    return jsonify(novo_filme), 201

In [ ]:
@app.route('/filmes/<int:id_filme>', methods=['PUT'])
def atualizar_filme(id_filme):

    filme = buscar_filme_por_id(id_filme)

    if not filme:
        return jsonify({"erro":"Filme não encontrado"}), 404

    dados = request.get_json()

    filme["titulo"] = dados["titulo"]
    filme["duracao"] = dados["duracao"]
    filme["genero"] = dados["genero"]
    filme["sinopse"] = dados["sinopse"]
    filme["nota_imdb"] = dados["nota_imdb"]

    salvar_filmes()

    return jsonify(filme)

In [ ]:
@app.route('/filmes/<int:id_filme>', methods=['PATCH'])
def atualizar_parcial(id_filme):

    filme = buscar_filme_por_id(id_filme)

    if not filme:
        return jsonify({"erro":"Filme não encontrado"}), 404

    dados = request.get_json()

    for chave, valor in dados.items():
        filme[chave] = valor

    salvar_filmes()

    return jsonify(filme)

In [ ]:
@app.route('/filmes/<int:id_filme>', methods=['DELETE'])
def deletar_filme(id_filme):

    filme = buscar_filme_por_id(id_filme)

    if not filme:
        return jsonify({"erro":"Filme não encontrado"}), 404

    filmes.remove(filme)

    salvar_filmes()

    return jsonify({
        "mensagem":"Filme removido"
    })

In [ ]:
app.run(debug=False, port=5003)

# Testes com requests

In [ ]:
import requests

base_url = "http://127.0.0.1:5001" 

In [ ]:
requests.get(f"{base_url}/filmes").json()

In [ ]:
novo = {
    "titulo":"Filme Teste",
    "duracao":120,
    "genero":"Drama",
    "sinopse":"Filme criado via POST",
    "nota_imdb":8.0
}

requests.post(
    f"{base_url}/filmes",
    json=novo
).json()

In [ ]:
requests.patch(
    f"{base_url}/filmes/1",
    json={"nota_imdb":9.9}
).json()